# 🎵 Music Search with CLAP + Weaviate

Build a semantic music search engine that understands natural language queries like:
- "chill lo-fi beats for studying"
- "aggressive metal with fast drums"
- "90s hip hop with jazz samples"

## How it works

1. **CLAP** (Contrastive Language-Audio Pretraining) embeds both text and audio in the same vector space
2. **Weaviate** stores the embeddings and enables fast similarity search
3. Your text query gets embedded → finds nearest audio embeddings → returns matching songs

## Prerequisites

- Google account (you're already here!)
- Weaviate Cloud account (free sandbox): https://console.weaviate.cloud/

---

## Step 1: Setup Environment

First, let's enable GPU and install the required packages.

**Important:** Go to `Runtime → Change runtime type → T4 GPU` before running this notebook!

In [ ]:
# Install required packages (takes ~1-2 minutes)
!pip install -q datasets transformers torch weaviate-client

print("✅ Packages installed!")

In [ ]:
# Check if GPU is available
import torch

if torch.cuda.is_available():
    device = "cuda"
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
else:
    device = "cpu"
    print("⚠️ No GPU detected. Go to Runtime → Change runtime type → T4 GPU")
    print("   (CPU will work but be slower)")

## Step 2: Set Up Weaviate Cloud

Get your free Weaviate sandbox:

1. Go to https://console.weaviate.cloud/
2. Sign up / Log in
3. Click **Create Cluster** → Choose **Sandbox** (free)
4. Wait for it to be ready (~1 min)
5. Click on your cluster → **API Keys** → Copy the key
6. Copy the **Cluster URL** (looks like `https://xxx.weaviate.network`)

### Option A: Use Colab Secrets (Recommended)
1. Click the 🔑 key icon in the left sidebar
2. Add two secrets:
   - `WEAVIATE_URL` = your cluster URL
   - `WEAVIATE_API_KEY` = your API key
3. Toggle "Notebook access" ON for both

### Option B: Paste directly below

In [ ]:
# Try to get credentials from Colab Secrets first, fall back to manual input
try:
    from google.colab import userdata
    WEAVIATE_URL = userdata.get('WEAVIATE_URL')
    WEAVIATE_API_KEY = userdata.get('WEAVIATE_API_KEY')
    print("✅ Loaded credentials from Colab Secrets!")
except:
    print("Colab Secrets not found. Please enter manually:")
    WEAVIATE_URL = None
    WEAVIATE_API_KEY = None

# Manual fallback - paste your values here if Secrets don't work
if not WEAVIATE_URL or not WEAVIATE_API_KEY:
    # ⬇️ PASTE YOUR VALUES HERE ⬇️
    WEAVIATE_URL = ""  # e.g., "https://my-sandbox-abc123.weaviate.network"
    WEAVIATE_API_KEY = ""  # e.g., "your-api-key-here"
    # ⬆️ PASTE YOUR VALUES HERE ⬆️

    if WEAVIATE_URL and WEAVIATE_API_KEY:
        print("✅ Using manually entered credentials")
    else:
        print("❌ Please add your credentials above or use Colab Secrets")

## Step 3: Load the CLAP Model

CLAP (from LAION) is like CLIP but for audio:
- **CLIP**: Text ↔ Image (OpenAI)
- **CLAP**: Text ↔ Audio (LAION, open source, FREE)

Both text and audio get embedded into the same 512-dimensional vector space.

In [ ]:
from transformers import ClapModel, ClapProcessor

print("Loading CLAP model (first run downloads ~600MB)...")

model = ClapModel.from_pretrained("laion/clap-htsat-unfused")
processor = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")
model = model.to(device)
model.eval()

print("✅ CLAP model loaded!")

## Step 4: Load MusicCaps Dataset

Google's MusicCaps contains 5,521 music clips with human-written captions:
- 10-second clips from YouTube
- Expert captions describing the music
- Genre/mood labels

Example caption: *"A low quality recording of an acoustic guitar playing a sad melody with a slow tempo."*

In [ ]:
from datasets import load_dataset

print("Loading MusicCaps dataset...")
dataset = load_dataset("google/MusicCaps", split="train")

print(f"✅ Loaded {len(dataset)} tracks")
print(f"\nColumns: {dataset.column_names}")
print(f"\nExample caption:\n'{dataset[0]['caption']}'")

## Step 5: Generate CLAP Embeddings

We'll embed the captions using CLAP's text encoder. Since CLAP puts text and audio in the same space, searching with text will find matching audio.

**Note:** For production, you'd embed the actual audio files. We're using captions here for speed.

In [ ]:
import numpy as np
from tqdm import tqdm

# How many tracks to process (start small, increase later)
LIMIT = 1000  # @param {type:"slider", min:100, max:5521, step:100}

tracks = []

print(f"Generating embeddings for {LIMIT} tracks...\n")

for i in tqdm(range(min(LIMIT, len(dataset)))):
    row = dataset[i]
    caption = row.get("caption", "")

    if not caption:
        continue

    # Embed caption with CLAP
    inputs = processor(text=[caption], return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        embedding = model.get_text_features(**inputs)

    # Normalize (important for cosine similarity)
    embedding = embedding[0].cpu().numpy()
    embedding = embedding / np.linalg.norm(embedding)

    tracks.append({
        "caption": caption[:500],
        "ytid": row.get("ytid", ""),
        "youtube_url": f"https://youtube.com/watch?v={row.get('ytid', '')}&t={int(row.get('start_s', 0))}",
        "labels": row.get("audioset_positive_labels", ""),
        "embedding": embedding.tolist()
    })

print(f"\n✅ Generated {len(tracks)} embeddings")
print(f"   Dimensions: {len(tracks[0]['embedding'])}")

## Step 6: Connect to Weaviate & Create Collection

Now we'll store these embeddings in Weaviate for fast similarity search.

In [ ]:
import weaviate
from weaviate.classes.init import Auth
from weaviate.classes.config import Configure, Property, DataType

# Connect to Weaviate Cloud
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY)
)

print(f"✅ Connected to Weaviate!")
print(f"   Ready: {client.is_ready()}")

In [ ]:
COLLECTION_NAME = "MusicTrack"

# Delete existing collection if it exists (for clean reruns)
if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)
    print(f"Deleted existing '{COLLECTION_NAME}' collection")

# Create collection with custom vectors (we provide our own CLAP embeddings)
collection = client.collections.create(
    name=COLLECTION_NAME,
    vectorizer_config=Configure.Vectorizer.none(),  # We provide vectors
    properties=[
        Property(name="caption", data_type=DataType.TEXT),
        Property(name="ytid", data_type=DataType.TEXT),
        Property(name="youtube_url", data_type=DataType.TEXT),
        Property(name="labels", data_type=DataType.TEXT),
    ]
)

print(f"✅ Created collection '{COLLECTION_NAME}'")

In [ ]:
# Upload tracks with embeddings
from weaviate.classes.data import DataObject

print(f"Uploading {len(tracks)} tracks...")

collection = client.collections.get(COLLECTION_NAME)

# Batch insert for speed
with collection.batch.dynamic() as batch:
    for i, track in enumerate(tqdm(tracks)):
        batch.add_object(
            properties={
                "caption": track["caption"],
                "ytid": track["ytid"],
                "youtube_url": track["youtube_url"],
                "labels": track["labels"],
            },
            vector=track["embedding"]
        )

# Verify
count = collection.aggregate.over_all(total_count=True).total_count
print(f"\n✅ Uploaded {count} tracks to Weaviate!")

## Step 7: Search! 🎵

Now the fun part - search for music using natural language!

In [ ]:
def search_music(query: str, top_k: int = 5):
    """Search for music using a text description."""

    # Embed the query with CLAP
    inputs = processor(text=[query], return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        query_embedding = model.get_text_features(**inputs)

    # Normalize
    query_embedding = query_embedding[0].cpu().numpy()
    query_embedding = query_embedding / np.linalg.norm(query_embedding)

    # Search Weaviate
    collection = client.collections.get(COLLECTION_NAME)
    results = collection.query.near_vector(
        near_vector=query_embedding.tolist(),
        limit=top_k,
        return_metadata=["distance"]
    )

    # Display results
    print("=" * 70)
    print(f"🔍 Results for: \"{query}\"")
    print("=" * 70)

    for i, obj in enumerate(results.objects, 1):
        props = obj.properties
        # Convert distance to similarity (Weaviate uses distance, lower = better)
        similarity = 1 - (obj.metadata.distance or 0)

        print(f"\n{i}. Similarity: {similarity:.4f}")
        print(f"   📝 {props.get('caption', 'No caption')[:100]}...")
        if props.get('youtube_url'):
            print(f"   🎬 {props['youtube_url']}")

    print("\n" + "=" * 70)
    return results

In [ ]:
# Try some searches!
search_music("chill lo-fi beats for studying")

In [ ]:
search_music("aggressive metal with fast drums")

In [ ]:
search_music("upbeat electronic dance music")

In [ ]:
search_music("sad piano ballad")

In [ ]:
# 🎯 Try your own search!
YOUR_QUERY = "jazz with saxophone"  # @param {type:"string"}

search_music(YOUR_QUERY)

## Step 8: Cleanup (Optional)

Close the Weaviate connection when done.

In [ ]:
# Close connection when done
client.close()
print("✅ Weaviate connection closed")

---

## 🎉 Congratulations!

You've built a semantic music search engine using:

1. **CLAP** - Open source model that embeds text and audio in the same space (LAION, FREE)
2. **MusicCaps** - Google's dataset of music with captions
3. **Weaviate** - Vector database for fast similarity search

### Key Concepts

- **Embeddings**: Convert text/audio into numerical vectors that capture meaning
- **Vector similarity**: Similar meanings → vectors close together
- **Cross-modal search**: Search audio using text (or vice versa)

### Next Steps

- Increase `LIMIT` to index all 5,521 tracks
- Try embedding actual audio files (not just captions)
- Add filters (by genre, mood, etc.)
- Build a web UI with Gradio or Streamlit
- Explore Weaviate's hybrid search (combine vector + keyword)

### Resources

- [CLAP Paper](https://arxiv.org/abs/2211.06687)
- [LAION CLAP on Hugging Face](https://huggingface.co/laion/clap-htsat-unfused)
- [MusicCaps Dataset](https://huggingface.co/datasets/google/MusicCaps)
- [Weaviate Docs](https://weaviate.io/developers/weaviate)